# Veritas Agent — Evaluation Notebook

Interactive walkthrough of the pipeline: corpus ingestion, retrieval anatomy, a query
sandbox, the full 30-question benchmark with a real ablation ladder, and a four-panel
dashboard.

**Read this first.** Every number in this notebook is computed by running the code in it.
Nothing is hardcoded. Where the pipeline cannot be measured — specifically Gate B, which
needs a language model to produce claims — the notebook says so instead of filling the
gap with a plausible-looking figure.

Run order matters: cells assume the ones above have run.

## 0. Setup

In [ ]:
import os, sys, json, warnings
warnings.filterwarnings("ignore")

# Run from the repository root so the default relative config paths resolve.
if os.path.basename(os.getcwd()) != "Research agent" and os.path.exists("../config.yaml"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from veritas.config import load_config

cfg = load_config("config.yaml")
print(f"corpus      : {cfg.corpus_dir}")
print(f"embedding   : {cfg.embedding.model}")
print(f"reranker    : {cfg.rerank.model} (enabled={cfg.rerank.enabled})")
print(f"verifier    : {cfg.verify.model} (threshold={cfg.verify.support_threshold})")
print(f"Gate A tau  : {cfg.abstain.min_rerank_score}")
print(f"chunk target: {cfg.chunk.target_tokens} tokens, {cfg.chunk.overlap_sentences} unit overlap")

## 1. Corpus ingestion and the citation-integrity invariant

Every chunk stores the exact character span it was cut from. The whole system rests on
that: a citation that cannot be sliced back out of the source document is not a citation.
The assertion below is the same one in `test_veritas.py`.

In [ ]:
from veritas.ingest import run_ingest, load_file_content
from veritas.index import load_chunks

n_chunks, n_docs = run_ingest(cfg)
print(f"Ingested {n_docs} documents into {n_chunks} chunks\n")

chunks = load_chunks(cfg)
manifest = json.load(open(cfg.manifest_path, encoding="utf-8"))
sources = {d["doc_id"]: load_file_content(os.path.join(cfg.corpus_dir, d["filename"]))
           for d in manifest}

mismatches = [c.chunk_id for c in chunks
              if sources[c.doc_id][c.char_start:c.char_end] != c.text]
assert not mismatches, f"char offsets do not slice back: {mismatches}"
print(f"OK: all {len(chunks)} chunks slice back to their source document exactly.")

pd.DataFrame([{
    "chunk_id": c.chunk_id,
    "doc": c.doc_title[:28],
    "chars": c.char_end - c.char_start,
    "words": len(c.text.split()),
    "preview": c.text.replace("\n", " ")[:70] + "...",
} for c in chunks])

## 2. Retrieval anatomy

One query, every stage side by side: what dense retrieval found, what BM25 found, how RRF
fused them, and how the cross-encoder reordered the result.

In [ ]:
from veritas.index import search_hybrid
from veritas.rerank import rerank_chunks

QUERY = "What function in OpenSSH was intercepted by the XZ backdoor payload?"

fused, rrf_scores, dense_rank, bm25_rank = search_hybrid(QUERY, cfg, chunks=chunks)
reranked = rerank_chunks(QUERY, fused, cfg)

rrf_map = dict(rrf_scores)
rerank_map = {c.chunk_id: s for c, s in reranked}

stages = pd.DataFrame({
    "dense_rank":  {cid: i + 1 for i, cid in enumerate(dense_rank)},
    "bm25_rank":   {cid: i + 1 for i, cid in enumerate(bm25_rank)},
    "rrf_score":   rrf_map,
    "rerank_score": rerank_map,
}).sort_values("rrf_score", ascending=False)

print(f"Query: {QUERY}")
print(f"Context window sent to the generator: {[c.chunk_id for c, _ in reranked]}\n")
stages.head(10)

Note how far the cross-encoder moves things. RRF ranks by rank-position consensus and is
blind to *how* relevant a passage is; the cross-encoder reads the query and the passage
together and produces a calibrated relevance probability. That probability is what Gate A
thresholds on — a rank-position score could not be thresholded meaningfully.

## 3. Query sandbox

Three questions covering the three buckets in the benchmark. Change `QUESTIONS` and re-run.

In [ ]:
from veritas.pipeline import run_pipeline

QUESTIONS = [
    ("answerable",   "What functions were targeted in the XZ Utils backdoor?"),
    ("unanswerable", "What is the capital city of France?"),
    ("adversarial",  "What is the exact CVSS score of TLS 1.3 RFC 8446?"),
]

for bucket, question in QUESTIONS:
    trace = run_pipeline(question, cfg, offline=True)
    print("=" * 100)
    print(f"[{bucket}] {question}")
    print("=" * 100)
    if trace.abstained:
        print("ABSTAINED\n" + trace.abstain_reason)
    else:
        for claim, verdict in zip(trace.final_answer.claims, trace.verdicts):
            print(f"CLAIM      : {claim.text[:220].replace(chr(10), ' ')}...")
            print(f"CITATIONS  : {claim.citations}")
            print(f"ENTAILMENT : {verdict.score:.3f} (supported={verdict.supported}) "
                  f"via {verdict.backend}")
        print(f"GENERATOR  : {trace.final_answer.generator}")
    print()

The `GENERATOR` line is the one to watch. `extractive-fallback` means no language model
was reachable and the agent quoted the top-ranked passage verbatim. Such an answer entails
itself trivially, so its entailment score says nothing about verification quality — and
the adversarial question above is answered rather than refused for exactly that reason.

Set `GEMINI_API_KEY` or `OPENROUTER_API_KEY`, or run a local Ollama model, and re-run with
`offline=False` to exercise the real path.

## 4. Benchmark and ablation ladder

30 annotated questions: 15 answerable, 5 unanswerable (topic absent from the corpus), 10
adversarial (topic present, requested fact absent).

Four configurations, all really executed — the weaker columns are the same pipeline with
stages switched off through config overrides, not remembered constants.

In [ ]:
from eval.run_eval import execute_eval, ABLATIONS

results = execute_eval("eval/gold.jsonl", cfg, offline=True, ablations=True)

In [ ]:
rows = []
for name, r in results.items():
    rows.append({
        "variant": name,
        "abstention_accuracy": r["abstention"]["accuracy"],
        "false_answer_rate": r["abstention"]["false_answer_rate"],
        "over_refusal_rate": r["abstention"]["over_refusal_rate"],
        "gold_citation_precision": r["gold_citation"]["precision"],
        "gold_citation_recall": r["gold_citation"]["recall"],
        "gold_citation_f1": r["gold_citation"]["f1"],
        "retrieval_recall_at_k": r["retrieval"]["recall_at_final_k"],
        "retrieval_mrr": r["retrieval"]["mrr"],
        "auroc": r["auroc_signal"],
    })
summary = pd.DataFrame(rows).set_index("variant")
summary.T

## 5. Gate A signal and threshold trade-off

The reranker's maximum score is the pre-generation abstention signal. This is the honest
picture of how well it separates answerable from unanswerable questions.

In [ ]:
from eval.calibrate import collect_signals

signals = pd.DataFrame(collect_signals(cfg, "eval/gold.jsonl"))
signals.groupby("bucket")["max_score"].describe()[["count", "min", "mean", "max"]]

In [ ]:
taus = np.linspace(0, 1, 201)
sweep = []
for tau in taus:
    fa = ((~signals.is_answerable) & (signals.max_score >= tau)).sum()
    tr = ((~signals.is_answerable) & (signals.max_score <  tau)).sum()
    fr = (( signals.is_answerable) & (signals.max_score <  tau)).sum()
    ta = (( signals.is_answerable) & (signals.max_score >= tau)).sum()
    sweep.append({"tau": tau,
                  "far": fa / (fa + tr) if (fa + tr) else 0.0,
                  "orr": fr / (fr + ta) if (fr + ta) else 0.0,
                  "accuracy": (ta + tr) / len(signals)})
sweep = pd.DataFrame(sweep)
print(f"At the configured tau={cfg.abstain.min_rerank_score}: "
      f"FAR={sweep.iloc[(sweep.tau - cfg.abstain.min_rerank_score).abs().idxmin()].far:.3f}, "
      f"ORR={sweep.iloc[(sweep.tau - cfg.abstain.min_rerank_score).abs().idxmin()].orr:.3f}")
sweep.iloc[::25]

## 6. Dashboard

Four panels, each answering one question:

1. **Where do the decisions land?** — answer/refuse confusion matrix for the full pipeline.
2. **What does each stage buy?** — the ablation ladder.
3. **Can the Gate A signal separate the buckets?** — per-question reranker scores.
4. **What does moving the threshold cost?** — the FAR/ORR trade-off curve.

In [ ]:
# Categorical slots from a CVD-validated palette (worst all-pairs deutan dE 9.2,
# normal-vision dE 24.0 on a light surface). Every mark is also directly labelled,
# so identity never depends on color alone.
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, INK_SOFT, GRID = "#0b0b0b", "#52514e", "#dcdcd8"
SURFACE = "#fcfcfb"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK_SOFT,
    "text.color": INK, "xtick.color": INK_SOFT, "ytick.color": INK_SOFT,
    "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": GRID, "grid.linewidth": 0.8, "font.size": 9,
})

fig, axes = plt.subplots(2, 2, figsize=(13.5, 9.5))
fig.suptitle("Veritas Agent — measured evaluation, 30 gold questions",
             fontsize=13, fontweight="bold", x=0.02, ha="left", y=0.985)

# ---- Panel 1: confusion matrix (sequential single hue, light -> dark = magnitude) ----
ax = axes[0][0]
cm_stats = results["veritas_full"]["abstention"]
matrix = np.array([[cm_stats["ta"], cm_stats["fr"]],
                   [cm_stats["fa"], cm_stats["tr"]]])
ax.imshow(matrix, cmap="Blues", vmin=0, vmax=matrix.max())
for (i, j), value in np.ndenumerate(matrix):
    ax.text(j, i, str(value), ha="center", va="center", fontsize=22, fontweight="bold",
            color="#ffffff" if value > matrix.max() * 0.55 else INK)
ax.set_xticks([0, 1], ["agent answered", "agent refused"])
ax.set_yticks([0, 1], ["answerable", "unanswerable"])
ax.set_title("1 · Where the decisions land (full pipeline)", fontweight="bold", loc="left")
ax.set_xlabel(f"FAR {cm_stats['false_answer_rate']:.1%}   ORR {cm_stats['over_refusal_rate']:.1%}"
              f"   accuracy {cm_stats['accuracy']:.1%}")
ax.grid(False)
for spine in ax.spines.values():
    spine.set_visible(False)

# ---- Panel 2: ablation ladder (grouped bars, 3 series, direct-labelled) ----
ax = axes[0][1]
variants = list(results.keys())
series = [("Abstention accuracy", "abstention_accuracy", BLUE),
          ("Gold citation F1", "gold_citation_f1", ORANGE),
          ("Abstention AUROC", "auroc", AQUA)]
x = np.arange(len(variants))
width = 0.26
for k, (label, column, color) in enumerate(series):
    values = summary[column].values
    offset = (k - 1) * (width + 0.02)   # 2px-equivalent gap between adjacent fills
    bars = ax.bar(x + offset, values, width, label=label, color=color, zorder=3)
    for bar, value in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, value + 0.02, f"{value:.2f}",
                ha="center", fontsize=7.5, color=INK_SOFT)
ax.set_xticks(x, [v.replace("_", "\n") for v in variants])
ax.set_ylim(0, 1.15)
ax.set_ylabel("score")
ax.set_title("2 · What each stage buys", fontweight="bold", loc="left")
ax.legend(frameon=False, fontsize=8, ncol=1, loc="upper left")
ax.yaxis.grid(True, zorder=0)

# ---- Panel 3: Gate A signal per question, by bucket ----
ax = axes[1][0]
bucket_color = {"answerable": BLUE, "unanswerable": ORANGE, "adversarial": AQUA}
rng = np.random.default_rng(0)
for i, (bucket, color) in enumerate(bucket_color.items()):
    subset = signals[signals.bucket == bucket]
    jitter = rng.uniform(-0.13, 0.13, len(subset))
    ax.scatter(i + jitter, subset.max_score, s=58, color=color, zorder=3,
               edgecolor=SURFACE, linewidth=1.6)   # surface ring on overlapping marks
    ax.text(i, 1.06, f"n={len(subset)}", ha="center", fontsize=8, color=INK_SOFT)
ax.axhline(cfg.abstain.min_rerank_score, color=INK_SOFT, linestyle="--", linewidth=1.2, zorder=2)
ax.text(2.48, cfg.abstain.min_rerank_score + 0.02, f"Gate A tau = {cfg.abstain.min_rerank_score}",
        ha="right", fontsize=8, color=INK_SOFT)
ax.set_xticks(range(3), list(bucket_color))
ax.set_ylim(-0.05, 1.12)
ax.set_ylabel("max cross-encoder score")
ax.set_title("3 · Gate A signal — adversarial questions overlap answerable ones",
             fontweight="bold", loc="left")
ax.yaxis.grid(True, zorder=0)

# ---- Panel 4: threshold trade-off (2 lines, one shared 0-1 axis) ----
ax = axes[1][1]
ax.plot(sweep.tau, sweep.far, color=BLUE, linewidth=2, zorder=3)
ax.plot(sweep.tau, sweep.orr, color=ORANGE, linewidth=2, zorder=3)
ax.text(0.30, 0.62, "false answer rate", color=BLUE, fontsize=9, fontweight="bold")
ax.text(0.62, 0.14, "over-refusal rate", color=ORANGE, fontsize=9, fontweight="bold")
ax.axvline(cfg.abstain.min_rerank_score, color=INK_SOFT, linestyle="--", linewidth=1.2, zorder=2)
ax.text(cfg.abstain.min_rerank_score + 0.015, 1.02, "configured", fontsize=8, color=INK_SOFT)
ax.set_xlabel("Gate A threshold (tau)")
ax.set_ylabel("rate")
ax.set_ylim(-0.03, 1.08)
ax.set_title("4 · Threshold trade-off — no setting is free", fontweight="bold", loc="left")
ax.yaxis.grid(True, zorder=0)

fig.tight_layout(rect=[0, 0.03, 1, 0.96])

generators = results["veritas_full"]["generators"]
fig.text(0.02, 0.005,
         f"Generators used: {generators}. Extractive-fallback answers cannot abstain and "
         f"entail themselves, so panels 1 and 2 understate what Gate B contributes.",
         fontsize=8, color=INK_SOFT)
plt.show()

## 7. What these results do and do not show

**Valid as shown, independent of the generator**

- Retrieval recall@6 and MRR against annotated gold chunks.
- The Gate A signal distribution and its AUROC.
- The threshold trade-off curve.
- Chunk/citation span integrity (section 1).

**Not valid as shown**

- End-to-end abstention accuracy, FAR and ORR, whenever the `generators` line above
  reports `extractive-fallback`. That fallback quotes the top passage verbatim: it has no
  way to decide a passage fails to answer the question, so it never abstains.
- The apparent equality of `plus_reranker` and `veritas_full`. Gate B rejects claims that
  their cited chunks do not entail; a verbatim quote always entails itself, so Gate B has
  nothing to reject. Its contribution is **unmeasured here, not zero**.

**The measured limitation of the architecture**

Panel 3 is the real finding: seven of the ten adversarial questions retrieve a genuinely
on-topic passage scoring as high as an answerable question. Retrieval relevance alone
cannot tell "this passage answers the question" from "this passage is about the right
topic but lacks the fact". No threshold fixes that — it needs the generation-time and
verification-time checks, which need a language model.

**Other caveats**

- The benchmark is self-authored alongside the corpus; 30 questions means one question is
  3.3 percentage points.
- `gold_answer_token_recall` is a bag-of-words proxy for answer correctness, not human
  grading.